# Quantum-Agility Auditor — Colab Training Notebook

**Use this notebook for TRAINING ONLY.**

The dashboard (Module 4) is a Streamlit app and the scanners (Module 1) walk a
local directory tree — neither works well in Colab. Train here, download the two
`.joblib` files, drop them into `models/` on your machine, and run
`streamlit run app.py` locally.

Runtime needed: **CPU is fine.** No GPU required. Total ~3–5 minutes.


## 1. Upload the project

Zip the `quantum-agility-auditor` folder on your machine and upload it here.

In [ ]:
from google.colab import files
import zipfile, pathlib, shutil

uploaded = files.upload()          # select quantum-agility-auditor.zip
name = next(iter(uploaded))
if pathlib.Path('quantum-agility-auditor').exists():
    shutil.rmtree('quantum-agility-auditor')
with zipfile.ZipFile(name) as z:
    z.extractall('.')
print('extracted:', sorted(p.name for p in pathlib.Path('quantum-agility-auditor').iterdir()))

## 2. Install dependencies

Streamlit and Scapy are skipped — they aren't needed for training.

In [ ]:
%pip install -q scikit-learn numpy scipy pandas joblib lime cryptography
print('dependencies ready')

## 3. Inspect the corpus

Before training, look at what the generator actually produces.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path('quantum-agility-auditor').resolve()
sys.path.insert(0, str(ROOT))

from qaa.module2_detector.corpus import build_corpus, corpus_summary

rows = build_corpus()
s = corpus_summary(rows)
print(f"{s['total']} snippets | {s['classes']} classes | "
      f"{s['template_families']} template families | {len(s['by_language'])} languages")
print('\nper class:', s['by_label'])
print('\nper language:', s['by_language'])
print('\nobfuscation mix:', s['by_obfuscation'])

In [ ]:
# A few samples, including the obfuscated ones the classifier exists to catch.
for r in rows[:4]:
    print(f"--- {r['label']} | {r['template_family']} | {r['obfuscation']} ---")
    print(r['text'])
    print()

## 4. Train both models

This runs three evaluation protocols. Read the notes it prints — the random-split
number is **not** the one to quote.

In [ ]:
import os
os.chdir(ROOT)
!python scripts/train_all.py --lofo-sample 40

## 5. Review the metrics

In [ ]:
import json, pandas as pd
m = json.loads((ROOT / 'models' / 'nlp_metrics.json').read_text())

rows = []
for key, nice in [('random_split','Random split (optimistic)'),
                  ('lofo','Leave-one-library-out (HEADLINE)'),
                  ('group_kfold','Grouped 4-fold (lower bound)')]:
    p = m['protocols'].get(key, {})
    if p.get('skipped'): continue
    rows.append({'Protocol': nice, 'Accuracy': p.get('accuracy'),
                 'Precision': p.get('precision_macro'), 'Recall': p.get('recall_macro'),
                 'F1': p.get('f1_macro'), 'FPR': p.get('false_positive_rate')})
display(pd.DataFrame(rows))

print('\nWeakest held-out libraries:')
print(m['protocols'].get('lofo', {}).get('weakest_families'))

In [ ]:
fp = json.loads((ROOT / 'models' / 'fingerprint_metrics.json').read_text())
print('Fingerprint accuracy:', fp['holdout']['accuracy'])
print()
print(fp['interpretation'])
print()
print('Top features (note how structural features dominate the SP 800-22 ones):')
for name, imp in fp['feature_importance_top15'][:10]:
    print(f'  {imp:.4f}  {name}')

## 6. Confusion matrix

Look at where the fingerprint classifier fails — AES-GCM vs ChaCha20 sitting near
chance is the expected and correct result, not a bug. See the README.

In [ ]:
import numpy as np, pandas as pd
labels = fp['labels']
cm = pd.DataFrame(np.array(fp['confusion_matrix']), index=labels, columns=labels)
display(cm)

## 7. Run the end-to-end benchmark

In [ ]:
!python scripts/evaluate.py

## 8. Run the test suite

In [ ]:
!python tests/test_modules.py

## 9. Download the trained models

Put both files in `models/` on your machine, then run:

```bash
streamlit run app.py
```

In [ ]:
from google.colab import files
files.download(str(ROOT / 'models' / 'nlp_classifier.joblib'))
files.download(str(ROOT / 'models' / 'fingerprint_classifier.joblib'))
files.download(str(ROOT / 'models' / 'nlp_metrics.json'))
files.download(str(ROOT / 'models' / 'fingerprint_metrics.json'))